In [ ]:
###
# 1. 環境のセットアップ
###

import sys
import os
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

ROOT_PATH = Path('/content/drive/MyDrive/cnn-hands-on')
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

os.chdir(ROOT_PATH)

# 日本語フォント対応
!pip install -q japanize-matplotlib
import japanize_matplotlib

print(f"✅ 環境セットアップ完了！現在のディレクトリ: {Path.cwd()}")

# 1. プーリング層とは

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torchvision.transforms as transforms
from PIL import ImageOps

# 画像を読み込む
img_path = ROOT_PATH / "data" / "cats_vs_dogs" / "Cat" / "00000.jpg"
img = Image.open(img_path)

# MaxPoolingの動作を視覚化
sample = np.array([
    [1, 3, 2, 1],
    [4, 2, 3, 1],
    [2, 4, 1, 2],
    [3, 1, 2, 3]
])

print("入力 (4x4):")
print(sample)
print()
print("MaxPooling (2x2) の結果:")
print("左上2x2: max([1,3,4,2]) =", np.max(sample[0:2, 0:2]))
print("右上2x2: max([2,1,3,1]) =", np.max(sample[0:2, 2:4]))
print("左下2x2: max([2,4,3,1]) =", np.max(sample[2:4, 0:2]))
print("右下2x2: max([1,2,2,3]) =", np.max(sample[2:4, 2:4]))

In [ ]:
# Conv → MaxPooling のアニメーション可視化
import torch
import torch.nn as nn
import matplotlib.patches as patches
from IPython.display import display
import time

# 画像準備
N = 128
img_small = transforms.Resize((N, N))(img)
img_gray = ImageOps.grayscale(img_small)
img_array = np.array(img_gray) / 255.0

# PyTorchテンソルに変換 (B, C, H, W)
x = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 畳み込み層（エッジ検出フィルタを手動設定）
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv.weight[0, 0] = torch.tensor([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1]
    ], dtype=torch.float32)

# 畳み込み実行
feature_map = conv(x).squeeze().detach().numpy()
print(f"Conv後の特徴マップ: {feature_map.shape}")

# MaxPoolingのアニメーション
pool_size = 2
h, w = feature_map.shape
out_h, out_w = h // pool_size, w // pool_size
output_array = np.zeros((out_h, out_w))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(feature_map, cmap='gray')
rect1 = patches.Rectangle((-0.5, -0.5), pool_size, pool_size, linewidth=2, edgecolor='red', facecolor='none')
ax1.add_patch(rect1)
ax1.set_title(f"Conv Feature Map ({h}x{w})")
ax1.axis('off')

im2 = ax2.imshow(output_array, cmap='gray', vmin=feature_map.min(), vmax=feature_map.max())
rect2 = patches.Rectangle((-0.5, -0.5), 1, 1, linewidth=2, edgecolor='red', facecolor='none')
ax2.add_patch(rect2)
ax2.set_title(f"MaxPooling Output ({out_h}x{out_w})")
ax2.axis('off')

display_handle = display(fig, display_id=True)
plt.close(fig)

time.sleep(1)

skip_steps = 10
step_count = 0

for y in range(out_h):
    for x in range(out_w):
        region = feature_map[y*pool_size:(y+1)*pool_size, x*pool_size:(x+1)*pool_size]
        output_array[y, x] = np.max(region)
        step_count += 1

        if step_count % skip_steps == 0 or (y == out_h-1 and x == out_w-1):
            rect1.set_xy((x*pool_size - 0.5, y*pool_size - 0.5))
            im2.set_data(output_array)
            rect2.set_xy((x - 0.5, y - 0.5))
            display_handle.update(fig)
            time.sleep(0.05)

# 2. プーリングの実装

In [ ]:
###
# 演習1: MaxPoolingを手動で実装
###

def max_pool2d(img, pool_size=2):
    # 入力画像の高さと幅を取得
    h, w = img.shape[:?]
    
    # 出力サイズを計算（入力サイズ // pool_size）
    out_h = h // ?
    out_w = w // ?
    
    # 出力配列を初期化
    out = np.zeros((?, ?))
    
    for i in range(out_h):
        for j in range(out_w):
            # pool_size × pool_size の領域を切り出す
            region = img[
                i*?:(i+1)*?,
                j*?:(j+1)*?
            ]
            # 最大値を取得
            out[i, j] = np.?(region)
    return out

In [ ]:
# テスト
test_img = np.array([
    [1, 3, 2, 1],
    [4, 2, 3, 1],
    [2, 4, 1, 2],
    [3, 1, 2, 3]
])

result = max_pool2d(test_img, pool_size=2)
print("入力:")
print(test_img)
print("\nMaxPooling結果:")
print(result)
print("\n期待される出力: [[4, 3], [4, 3]]")

---

In [ ]:
###
# 発展演習: AveragePoolingも実装
###

def avg_pool2d(img, pool_size=2):
    h, w = img.shape[:2]
    out_h = h // pool_size
    out_w = w // pool_size
    out = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = img[
                i*pool_size:(i+1)*pool_size,
                j*pool_size:(j+1)*pool_size
            ]
            # np.mean() で平均値を取得
            out[i, j] = np.?(region)
    return out

In [ ]:
# テスト
result_avg = avg_pool2d(test_img, pool_size=2)
print("入力:")
print(test_img)
print("\nAveragePooling結果:")
print(result_avg)
print("\n期待される出力: [[2.5, 1.75], [2.5, 2.0]]")

---

In [ ]:
# 実際の画像に適用
pooled_max = max_pool2d(feature_map, pool_size=2)
pooled_avg = avg_pool2d(feature_map, pool_size=2)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(feature_map, cmap='gray')
axes[0].set_title(f'Input ({feature_map.shape[0]}x{feature_map.shape[1]})')

axes[1].imshow(pooled_max, cmap='gray')
axes[1].set_title(f'MaxPooling ({pooled_max.shape[0]}x{pooled_max.shape[1]})')

axes[2].imshow(pooled_avg, cmap='gray')
axes[2].set_title(f'AvgPooling ({pooled_avg.shape[0]}x{pooled_avg.shape[1]})')

plt.tight_layout()
plt.show()

---

In [ ]:
###
# PyTorchでのプーリング
###

import torch
import torch.nn as nn

# MaxPooling層を定義
pool = nn.MaxPool2d(
    kernel_size=2,  # 領域サイズ
    stride=2        # 移動幅
)

# 推論 (B, C, H, W) 形式
x = torch.randn(1, 1, 28, 28)
output = pool(x)

print(f"入力サイズ: {x.shape}")
print(f"出力サイズ: {output.shape}")

# 3. 活性化関数とは

In [ ]:
# Conv → ReLU のアニメーション可視化
import torch
import torch.nn as nn
import matplotlib.patches as patches
from IPython.display import display
import time

# 画像準備（プーリングと同じ）
N = 64
img_small = transforms.Resize((N, N))(img)
img_gray = ImageOps.grayscale(img_small)
img_array = np.array(img_gray) / 255.0

# PyTorchテンソルに変換
x = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 畳み込み層（エッジ検出フィルタ）
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv.weight[0, 0] = torch.tensor([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1]
    ], dtype=torch.float32)

# 畳み込み実行
feature_map = conv(x).squeeze().detach().numpy()
print(f"Conv後の特徴マップ: {feature_map.shape}")
print(f"負の値の数: {np.sum(feature_map < 0)} / {feature_map.size}")

# ReLUのアニメーション
h, w = feature_map.shape
output_array = np.copy(feature_map)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

vmin, vmax = feature_map.min(), feature_map.max()
ax1.imshow(feature_map, cmap='RdBu_r', vmin=vmin, vmax=vmax)
ax1.set_title(f"Conv Feature Map (青=負, 赤=正)")
ax1.axis('off')

im2 = ax2.imshow(output_array, cmap='RdBu_r', vmin=vmin, vmax=vmax)
rect2 = patches.Rectangle((-0.5, -0.5), 1, 1, linewidth=2, edgecolor='green', facecolor='none')
ax2.add_patch(rect2)
ax2.set_title("ReLU Output (負→0)")
ax2.axis('off')

display_handle = display(fig, display_id=True)
plt.close(fig)

time.sleep(1)

skip_steps = 20
step_count = 0

for y in range(h):
    for x in range(w):
        # ReLU: 負の値を0に
        if feature_map[y, x] < 0:
            output_array[y, x] = 0
        step_count += 1

        if step_count % skip_steps == 0 or (y == h-1 and x == w-1):
            im2.set_data(output_array)
            rect2.set_xy((x - 0.5, y - 0.5))
            display_handle.update(fig)
            time.sleep(0.02)

print(f"ReLU後の負の値の数: {np.sum(output_array < 0)}")

# 正の値が変化していないことを確認
positive_mask = feature_map > 0
print(f"正の値は変化していない: {np.allclose(feature_map[positive_mask], output_array[positive_mask])}")

In [ ]:
# 代表的な活性化関数の可視化（特徴マップに適用）
import torch.nn.functional as F

# Conv後の特徴マップをPyTorchテンソルに
feature_tensor = torch.tensor(feature_map, dtype=torch.float32)

# 各活性化関数を適用
sigmoid_out = torch.sigmoid(feature_tensor).numpy()
tanh_out = torch.tanh(feature_tensor).numpy()
relu_out = F.relu(feature_tensor).numpy()

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Conv後の特徴マップ（入力）
axes[0, 0].imshow(feature_map, cmap='RdBu_r')
axes[0, 0].set_title(f'Conv Feature Map\n(値の範囲: {feature_map.min():.2f} ~ {feature_map.max():.2f})')
axes[0, 0].axis('off')

# Sigmoid適用後
axes[0, 1].imshow(sigmoid_out, cmap='gray')
axes[0, 1].set_title(f'Sigmoid (0~1)\n(値の範囲: {sigmoid_out.min():.2f} ~ {sigmoid_out.max():.2f})')
axes[0, 1].axis('off')

# Tanh適用後
axes[1, 0].imshow(tanh_out, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1, 0].set_title(f'Tanh (-1~1)\n(値の範囲: {tanh_out.min():.2f} ~ {tanh_out.max():.2f})')
axes[1, 0].axis('off')

# ReLU適用後
axes[1, 1].imshow(relu_out, cmap='gray')
axes[1, 1].set_title(f'ReLU（主流）\n(値の範囲: {relu_out.min():.2f} ~ {relu_out.max():.2f})')
axes[1, 1].axis('off')

plt.suptitle('活性化関数の比較: 同じ特徴マップに異なる活性化関数を適用', fontsize=14)
plt.tight_layout()
plt.show()

# 4. 活性化関数の実装

In [ ]:
###
# 演習2: ReLUを手動で実装
###

def relu(x):
    # np.maximum(a, b): 要素ごとに大きい方を返す
    return np.?(0, x)

In [ ]:
# テスト
test_x = np.array([-2, -1, 0, 1, 2])
result = relu(test_x)
print(f"入力: {test_x}")
print(f"ReLU結果: {result}")
print(f"期待される出力: [0, 0, 0, 1, 2]")

---

In [ ]:
###
# 別の実装方法（条件付き代入）
###

def relu_v2(x):
    out = np.copy(x)
    # 0以下の値を0に置き換える
    out[out ? 0] = 0
    return out

In [ ]:
# 特徴マップに適用
feature_map = np.array([
    [-1, 2, -3],
    [4, -5, 6]
])

print("入力特徴マップ:")
print(feature_map)
print("\nReLU後:")
print(relu(feature_map))
print("\n期待される出力: [[0, 2, 0], [4, 0, 6]]")

---

In [ ]:
###
# PyTorchでの活性化関数
###

import torch
import torch.nn as nn
import torch.nn.functional as F

# ReLU層を定義
relu_layer = nn.ReLU()

# 推論
x = torch.tensor([-2., -1., 0., 1., 2.])
output = relu_layer(x)
print(f"nn.ReLU: {output}")

# 関数としても使える
output2 = F.relu(x)
print(f"F.relu: {output2}")

# 5. Conv → ReLU → Pool パイプライン

In [ ]:
###
# 発展演習: CNNの1ブロックを組む
###

import torch
import torch.nn as nn

# 各層を定義
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# 入力（1枚、1ch、28×28）
x = torch.randn(1, 1, 28, 28)
print(f"入力: {x.shape}")

# 順番に適用
x = conv(x)
print(f"Conv後: {x.shape}")

x = relu(x)
print(f"ReLU後: {x.shape}")

x = pool(x)
print(f"Pool後: {x.shape}")

In [ ]:
# Conv → ReLU → Pool パイプラインの可視化
import torch
import torch.nn as nn

# 画像準備
N = 128
img_small = transforms.Resize((N, N))(img)
img_gray = ImageOps.grayscale(img_small)
img_array = np.array(img_gray) / 255.0

# PyTorchテンソルに変換 (B, C, H, W)
x = torch.tensor(img_array, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# 各層を定義
conv = nn.Conv2d(1, 1, kernel_size=3, padding=1, bias=False)
with torch.no_grad():
    conv.weight[0, 0] = torch.tensor([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1]
    ], dtype=torch.float32)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

# 各段階の出力を保存
x_input = x.squeeze().numpy()
x_conv = conv(x).squeeze().detach().numpy()
x_relu = relu(conv(x)).squeeze().detach().numpy()
x_pool = pool(relu(conv(x))).squeeze().detach().numpy()

# 可視化
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(x_input, cmap='gray')
axes[0].set_title(f'Input\n({x_input.shape[0]}x{x_input.shape[1]})')
axes[0].axis('off')

# Conv: 正負があるのでRdBu_r
axes[1].imshow(x_conv, cmap='RdBu_r')
axes[1].set_title(f'After Conv\n(値: {x_conv.min():.2f}~{x_conv.max():.2f})')
axes[1].axis('off')

# ReLU: 0以上なので、vmin=0, vmax=最大値で正規化
axes[2].imshow(x_relu, cmap='gray', vmin=0, vmax=x_relu.max())
axes[2].set_title(f'After ReLU\n(値: {x_relu.min():.2f}~{x_relu.max():.2f})')
axes[2].axis('off')

# Pool: 同様に正規化
axes[3].imshow(x_pool, cmap='gray', vmin=0, vmax=x_pool.max())
axes[3].set_title(f'After Pool\n(値: {x_pool.min():.2f}~{x_pool.max():.2f})')
axes[3].axis('off')

plt.suptitle('CNN 1ブロック: Conv → ReLU → Pool', fontsize=14)
plt.tight_layout()
plt.show()

print(f"入力: {x_input.shape} → Conv: {x_conv.shape} → ReLU: {x_relu.shape} → Pool: {x_pool.shape}")
print(f"ReLUで0になったピクセル: {np.sum(x_relu == 0)} / {x_relu.size} ({100*np.sum(x_relu == 0)/x_relu.size:.1f}%)")

In [ ]:
###
# 複数ブロックを重ねる
###

# 28x28 → 14x14 → 7x7
conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
relu = nn.ReLU()
pool = nn.MaxPool2d(kernel_size=2, stride=2)

x = torch.randn(1, 1, 28, 28)
print(f"入力: {x.shape}")

# Block 1
x = conv1(x)
x = relu(x)
x = pool(x)
print(f"Block1後: {x.shape}")

# Block 2
x = conv2(x)
x = relu(x)
x = pool(x)
print(f"Block2後: {x.shape}")

# 6. まとめ

- **プーリング層**: 特徴マップを縮小し、位置ズレに強くする。MaxPooling（最大値）が主流、学習パラメータなし
- **活性化関数**: 非線形性を加え、深いネットワークに意味を持たせる。ReLU（負→0、正→そのまま）が主流
- **CNNの基本ブロック**: `Conv → ReLU → Pool` の組み合わせが1セット

### 暇な人向け


In [ ]:
word = "JNPGCZBUXHJAVWXGWIZAXTIQYMRRSSYDNUWCJYVZVZZCYZYKWUMOJNZYUJIKCWXUVDDNOYJDXYIXADXJYZNZTSNQDXGUBYSZPRCRPQYIPTXCSIHNZXWFWSQKVYOHWIZJYWZDQSLPIFXRYWYLXWWYDCBWIKJQGWSUXPHCORZXSXLWWOIZPIMQXCWVCMAYWKKPRNWAYYATXCHQCZKTIWIRLOZVQWKXZGYRZUQJXDJQQYMYLNBZXWWMJXPZXKYPGWRETBPPDHUMQMKNUYHFGQKHMYKJKWYTIBZSTOZFHLQVYXLGCNIEXQFAGBWAFMXSWXTCWZKXSAXUZFLUYPWIGKWYUDTOOYYWZYQZXDVJSYSTGJWXNZGZOZSZCXCHZERWCIWYTIPQRWXZWCYYQYUWTNGZXZUBYKYVZWPEKOYZNWKYGPOYXLTWYYTAFYXPXXQWCWSZLMXRGKVCCWLANWWCBZYWLIRYGJRHMKWVBWXWGRLETQNZHYAQUTZK"

# 以下の操作をwordに対して行ってください
# 1. 全てのWを削除する
# 2. Xという文字の3つ先がXでないならその文字をEに変えてください
#   ex) XABC -> XABE, XABX -> XABX
# 3. 全てのEYという文字をXに変えてください
# 4. すべての操作を行った後の文字列を出力しなさい